# T-Lab RecSys 2026: Causal GPT-2 Sequential Recommender System
### End-to-End Training, Hyperparameter Tuning & Evaluation on Kaggle (T4 / P100 GPU)

This notebook trains a **Causal GPT-2 Transformer from scratch** for Next-Item Prediction on the **Yambda** dataset (Yandex Music).

**Pre-flight Checklist on Kaggle:**
1. In the right sidebar under **Settings -> Accelerator**, select **GPU T4 x 2** or **GPU P100**.
2. In **Settings -> Internet**, toggle **Internet ON**.
3. Attach your Yambda dataset under `/kaggle/input/` (via "+ Add Input" in the top right), or allow auto-download from Hugging Face.

## 1. Verify GPU Accelerator

In [ ]:
!nvidia-smi

import os
# Prevent CUDA memory fragmentation on long-running sequence training
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


## 2. Clone Repository & Setup Environment
Clones the production repository directly from GitHub so all modules (`data_utils.py`, `dataset.py`, `model.py`, `train_base.py`, `sweep.py`, `train_dpo.py`, `dpo_dataset.py`, `evaluate_test.py`, `reward_model.py`) appear natively in the Kaggle filesystem.

In [ ]:
import os
import sys

# Ensure CUDA expandable segments is enabled in current process
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

REPO_URL = "https://github.com/sdwqwt2/Music-Recommender-System.git"
REPO_DIR = "/kaggle/working/Music-Recommender-System"

# 1. Clone repository directly into Kaggle filesystem
if not os.path.exists(REPO_DIR):
    print(f"Cloning repository from {REPO_URL} into {REPO_DIR}...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repository already cloned at {REPO_DIR}. Pulling latest changes...")
    !cd {REPO_DIR} && git pull origin main

# 2. Switch working directory to repo root so all scripts work directly
%cd {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Active working directory: {os.getcwd()}")
!ls -lh


## 3. Install Dependencies

In [ ]:
!pip install --quiet -r requirements.txt
print("Dependencies installed successfully from requirements.txt.")


## 4. Optional: Weights & Biases (WandB) Setup
You can securely store your `WANDB_API_KEY` under Kaggle **Add-ons -> Secrets**.

In [ ]:
import os
USE_WANDB = False

try:
    from kaggle_secrets import UserSecretsClient
    import wandb
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    if wandb_key:
        wandb.login(key=wandb_key)
        print("[WandB] Successfully logged in via Kaggle Secrets!")
        USE_WANDB = True
    else:
        print("[WandB] WANDB_API_KEY secret not found. Running with local console logging.")
except Exception as e:
    print("[WandB] WandB not configured. Running with local console logging.")

## 5. Dataset Setup: Download Official Yambda or Use Mounted Input
Checks `/kaggle/input/` for attached datasets. If none is found, downloads **Yambda-50M** (`flat/50m/listens.parquet`) directly from Hugging Face.

In [ ]:
DOWNLOAD_REAL_YAMBDA = True

import os
import glob
from huggingface_hub import hf_hub_download

os.makedirs("/kaggle/working/data", exist_ok=True)
data_path = None
synthetic_flag = ""

# 1. Check if user attached a dataset via /kaggle/input/
input_files = glob.glob("/kaggle/input/**/*", recursive=True)
parquet_files = [f for f in input_files if f.endswith(".parquet") or f.endswith(".csv")]

if parquet_files:
    data_path = parquet_files[0]
    print(f"[Dataset] Found mounted dataset in /kaggle/input/: {data_path}")
elif DOWNLOAD_REAL_YAMBDA:
    print("[Dataset] Downloading official Yambda-50M from Hugging Face (~300MB)... ")
    try:
        data_path = hf_hub_download(
            repo_id="yandex/yambda",
            repo_type="dataset",
            filename="flat/50m/listens.parquet",
            local_dir="/kaggle/working/data"
        )
        print(f"[Dataset] Successfully downloaded Yambda to: {data_path}")
    except Exception as e:
        print(f"[Dataset] Download failed ({e}). Falling back to synthetic generator.")
        data_path = "/kaggle/working/data/synthetic_yambda.parquet"
        synthetic_flag = "--synthetic"
else:
    data_path = "/kaggle/working/data/synthetic_yambda.parquet"
    synthetic_flag = "--synthetic"
    print("[Dataset] Running in synthetic demo mode.")


## 6. Fast Hyperparameter Sweep (Recommended)
Instead of running 10 full epochs blindly (which takes ~5 hours), this sweep evaluates multiple configurations on a **20% user subsample** for 3 epochs each (~6 min/trial).

- Searches over: `n_embd` (128, 256, 384), `n_layer` (3, 4, 6), `n_head` (4, 8), `lr`, `warmup_ratio`, `weight_decay`
- Saves ranking table to `/kaggle/working/sweep_results/sweep_results.csv`
- Automatically saves the winning config to `/kaggle/working/sweep_results/best_config.json`

In [ ]:
# 1. Restore completed trials 1-4 so sweep.py skips directly to Trial 5
import os
import pandas as pd
from pathlib import Path

sweep_dir = Path("/kaggle/working/sweep_results")
sweep_dir.mkdir(parents=True, exist_ok=True)
csv_file = sweep_dir / "sweep_results.csv"

if not csv_file.exists():
    completed_trials = [
        {"trial_id": 1, "best_epoch": 3, "best_val_hr10": 0.1746, "best_val_ndcg10": 0.1026, "best_val_loss": 7.6193, "trial_duration_sec": 1698.0, "n_embd": 384, "n_layer": 3, "n_head": 4, "lr": 0.0003, "warmup_ratio": 0.03, "weight_decay": 0.1, "batch_size": 32, "gradient_accumulation_steps": 4},
        {"trial_id": 2, "best_epoch": 3, "best_val_hr10": 0.0504, "best_val_ndcg10": 0.0255, "best_val_loss": 9.3200, "trial_duration_sec": 912.7, "n_embd": 128, "n_layer": 3, "n_head": 4, "lr": 0.0001, "warmup_ratio": 0.03, "weight_decay": 0.01, "batch_size": 64, "gradient_accumulation_steps": 2},
        {"trial_id": 3, "best_epoch": 3, "best_val_hr10": 0.1881, "best_val_ndcg10": 0.1165, "best_val_loss": 7.6853, "trial_duration_sec": 1695.0, "n_embd": 384, "n_layer": 3, "n_head": 4, "lr": 0.001, "warmup_ratio": 0.1, "weight_decay": 0.05, "batch_size": 32, "gradient_accumulation_steps": 4},
        {"trial_id": 4, "best_epoch": 3, "best_val_hr10": 0.0983, "best_val_ndcg10": 0.0547, "best_val_loss": 8.5484, "trial_duration_sec": 953.9, "n_embd": 128, "n_layer": 4, "n_head": 4, "lr": 0.0003, "warmup_ratio": 0.03, "weight_decay": 0.05, "batch_size": 64, "gradient_accumulation_steps": 4},
    ]
    pd.DataFrame(completed_trials).to_csv(csv_file, index=False)
    print(f"[Restore] Restored trials 1-4 into {csv_file}. sweep.py will now resume directly from Trial 5!")

# 2. Run Hyperparameter Sweep (resumes at Trial 5, only runs remaining trials)
!python sweep.py \
    --data_path {data_path} \
    --processed_dir /kaggle/working/data/processed_sweep \
    --output_dir /kaggle/working/sweep_results \
    --num_trials 6 \
    --epochs_per_trial 3 \
    --subsample_users 0.20 \
    --max_items 100000 \
    --max_length 64 \
    --fp16 \
    --num_workers 2


## 7. Launch Full Training (With Best Sweep Config or Resuming Checkpoint)
This cell automatically reads the winning hyperparameters from `/kaggle/working/sweep_results/best_config.json` (if available) or defaults to the upgraded capacity architecture (`n_embd=256, n_layer=4, n_head=8, lr=5e-4, warmup_ratio=0.05`).

> **Resume Tip**: If you already have a saved checkpoint from a previous session, set `RESUME_CHECKPOINT = "/kaggle/working/checkpoints/best_gpt2_recsys.pt"` (or path from `/kaggle/input/...`) to resume weights directly!

In [ ]:
import json
import os

# Check if sweep generated a best config
sweep_cfg_path = "/kaggle/working/sweep_results/best_config.json"
if os.path.exists(sweep_cfg_path):
    with open(sweep_cfg_path, "r") as f:
        best_cfg = json.load(f)
    print(f"[Config] Loaded best config from sweep: {best_cfg}")
    n_embd = best_cfg.get("n_embd", 256)
    n_layer = best_cfg.get("n_layer", 4)
    n_head = best_cfg.get("n_head", 8)
    lr = best_cfg.get("lr", 5e-4)
    warmup = best_cfg.get("warmup_ratio", 0.05)
    wd = best_cfg.get("weight_decay", 0.01)
    bs = best_cfg.get("batch_size", 32)
    grad_accum = best_cfg.get("gradient_accumulation_steps", 4)
else:
    print("[Config] Using default upgraded capacity config")
    n_embd, n_layer, n_head = 256, 4, 8
    lr, warmup, wd = 5e-4, 0.05, 0.01
    bs, grad_accum = 32, 4

# Set path to resume weights from an existing checkpoint if desired
RESUME_CHECKPOINT = None  # e.g., "/kaggle/working/checkpoints/best_gpt2_recsys.pt"
resume_flag = f"--resume_from {RESUME_CHECKPOINT}" if RESUME_CHECKPOINT and os.path.exists(RESUME_CHECKPOINT) else ""
wandb_arg = "--use_wandb" if USE_WANDB else ""

# Launch full training with linear warmup, early stopping, and FP16
!python train_base.py \
    --data_path {data_path} \
    --processed_dir /kaggle/working/data/processed \
    --output_dir /kaggle/working/checkpoints \
    --user_col uid \
    --min_item_freq 5 \
    --max_items 100000 \
    --max_length 64 \
    --n_embd {n_embd} \
    --n_layer {n_layer} \
    --n_head {n_head} \
    --batch_size {bs} \
    --gradient_accumulation_steps {grad_accum} \
    --lr {lr} \
    --warmup_ratio {warmup} \
    --weight_decay {wd} \
    --early_stopping_patience 3 \
    --epochs 5 \
    --split_mode global_temporal \
    --fp16 \
    --num_workers 2 \
    {resume_flag} \
    {synthetic_flag} \
    {wandb_arg}


## 8. Dual-Track Model Verification & Qualitative Inspection
Runs dual-track evaluation on the held-out test set (comparing raw baseline vs engaged >=30s performance) and inspects real user trajectories.

In [ ]:
# Download the files into a local folder named 'downloaded_weights'
!kaggle kernels output alexrilenko/recsys-yambda -p ./downloaded_weights

In [ ]:
!unzip -q ./downloaded_weights/recsys_checkpoints.zip -d /kaggle/working/

In [ ]:
# Run full held-out test set evaluation and print 10 qualitative user recommendation samples
!python evaluate_test.py \
    --processed_dir /kaggle/working/downloaded_weights/data/processed \
    --checkpoint_path /kaggle/working/kaggle/working/checkpoints/best_gpt2_recsys.pt \
    --batch_size 32 \
    --num_samples 10 \
    --fp16


## 9. Phase 2 RL: Direct Preference Optimization (DPO) Policy Alignment
Directly aligns the pre-trained GPT-2 recommendation policy using pairwise offline preferences (engaged listens $\ge 30$s vs skips $< 30$s). Eliminates the value-network / critic approximation bottleneck entirely.

In [ ]:
!python train_dpo.py \
    --processed_dir /kaggle/working/downloaded_weights/data/processed \
    --base_checkpoint /kaggle/working/kaggle/working/checkpoints/best_gpt2_recsys.pt \
    --output_dir /kaggle/working/checkpoints \
    --epochs 5 \
    --batch_size 64 \
    --lr 5e-5 \
    --beta 0.1 \
    --sft_weight 0.1 \
    --skip_threshold_sec 30.0 \
    --gradient_accumulation_steps 2 \
    --fp16 \
    --num_workers 2


## 10. Held-Out Test Evaluation: DPO Policy vs Phase 1 Base Model
Computes Dual-Track metrics on `test_interactions.parquet` to verify whether DPO improved Engaged HR@10 and Duration-weighted NDCG while penalizing instant skips.

In [ ]:
# Evaluate DPO aligned policy on held-out test set
!python evaluate_test.py \
    --processed_dir /kaggle/working/downloaded_weights/data/processed \
    --checkpoint_path /kaggle/working/checkpoints/best_dpo_gpt2.pt \
    --batch_size 32 \
    --num_samples 10 \
    --fp16


## 11. Optional Alternative: State-Action Critic (Reward Model)
Trains an MLP Critic Q(s, a) for Top-K candidate re-ranking.

In [ ]:
!python reward_model.py \
    --processed_dir /kaggle/working/downloaded_weights/data/processed \
    --base_checkpoint /kaggle/working/kaggle/working/checkpoints/best_gpt2_recsys.pt \
    --output_dir /kaggle/working/checkpoints \
    --batch_size 64 \
    --lr 5e-4 \
    --skip_threshold_sec 30.0 \
    --max_duration_sec 180.0 \
    --epochs 5 \
    --fp16


## 12. Multi-Paradigm Benchmark, Statistical Significance & 5-Step Rollouts
Runs comprehensive comparative evaluation across:
1. **Base Causal GPT-2 (MLE)**
2. **2-Stage Actor-Critic Re-Ranking** ($\lambda \in \{0.1, 0.3, 0.5\}$)
3. **Direct Preference Optimization (DPO Policy)**
4. **Statistical Hypothesis Testing:** 5,000 Bootstrap 95% CIs, McNemar paired test ($\chi^2, p$), Wilcoxon signed-rank test
5. **Catalog Diversity:** Coverage %, Shannon Entropy, Gini Index
6. **5-Step Autoregressive Rollouts:** Exposure bias degradation across 5 unguided prediction steps.

In [ ]:
!python evaluate_analysis.py \n    --processed_dir /kaggle/working/downloaded_weights/data/processed \n    --base_checkpoint /kaggle/working/kaggle/working/checkpoints/best_gpt2_recsys.pt \n    --dpo_checkpoint /kaggle/working/checkpoints/best_dpo_gpt2.pt \n    --reward_checkpoint /kaggle/working/checkpoints/best_reward_model.pt \n    --output_json /kaggle/working/analysis_metrics.json \n    --num_bootstrap 5000 \n    --num_rollout_users 500


## 13. Optional Ablation: DPO Policy without SFT Regularization ($\gamma = 0.0$)
Tests the theoretical hypothesis: does removing the auxiliary SFT loss ($\gamma = 0.0$) cause language modeling degeneration and reward over-optimization?

In [ ]:
# Fast 2-minute ablation: evaluates policy drift when gamma (sft_weight) is set to 0.0
!python train_dpo.py \n    --processed_dir /kaggle/working/downloaded_weights/data/processed \n    --base_checkpoint /kaggle/working/kaggle/working/checkpoints/best_gpt2_recsys.pt \n    --output_dir /kaggle/working/checkpoints_no_sft \n    --sft_weight 0.0 \n    --epochs 3


## 14. Export Trained Artifacts & Analysis Results
Compresses Phase 1 Base Model, Phase 2 Critic & DPO checkpoints, and analysis results into a  archive for one-click download.

In [ ]:
!zip -r /kaggle/working/recsys_checkpoints.zip /kaggle/working/checkpoints /kaggle/working/analysis_metrics.json /kaggle/working/sweep_results /kaggle/working/data/processed
print("Artifacts archived successfully to: /kaggle/working/recsys_checkpoints.zip")
